# Behavioral Memory Architecture with ChromaDB
### Designing Tiered Vector Retrieval for Persistent AI Identity

**Author:** Mia L.  
**GitHub:** [miagonellm](https://github.com/miagonellm)  
**Stack:** ChromaDB · sentence-transformers · Python · Flask · Ollama

**Pinned Environment:**
```
chromadb==1.5.2
sentence-transformers==5.2.3
numpy==1.26.4
pandas==1.5.3
matplotlib==3.10.8
Flask==3.1.3
```

---

**Abstract**

In this document I'll be exploring memory retrieval architectures that are comprised of a multi-stage system; Short term, and long term memory encoded with compound metaddata 
tags that consolidates memory weight and impact. 

Designed and implemented a five-layer ChromaDB vector memory architecture with 29 compound emotional tags for persistent AI context retrieval. The system functions as a self-correcting feedback loop and creates enough retention for patterns in the model to structure. Curated exchanges are stored with emotional metadata, retrieved by semantic similarity at inference, and injected into the model prompt as behavioral anchors. The model's personality becomes partially constructed from what it retrieves — not just what's in its static system prompt.

Considering the prospects of what it takes for memory to adhere, and how data and pattern subsequently expositions that concept is always super fascinating to me.   


The result: an AI system whose behavior is consistent across sessions, adaptive to context, and shaped by curated historical interaction rather than prompt engineering alone.

---
## 1. Starting from base. 

Starting cold, LLMs have no memory. Every session starts from zero. The system prompt is static - it describes who the model should be, but it can't evolve, can't reference shared history, and can't adapt to relational context.

This creates three failures:

**1. Identity drift.** Without persistent context, you loose all progress in each session.

If the model's idenity is entirely prompt-dependent. Change the prompt slightly, the system changes. Restart the session, the system resets.

**2. Flat retrieval.** Standard RAG retrieves by semantic similarity alone. "What's most textually relevant" is not the same question as "what behavioral context should shape this response." An exchange about loss retrieved during a celebratory conversation poisons the tone.

**3. No feedback loop.** Most systems have no mechanism for the user to mark which interactions were good — which exchanges captured the model's system at its best. Without curation, retrieval quality degrades as the collection grows, because noise accumulates alongside signal.

This project addresses all three.

---
## 2. Architecture: Five-Layer Memory

The memory system is organized into five tiers, each serving a different function. They are queried in combination, not one-at-a-time - with results merged and weighted before prompt injection.

```
┌─────────────────────────────────────────────────┐
│              MEMORY ARCHITECTURE                 │
│                                                   │
│  Layer 1: conversations (active)                  │
│  ├── Rolling history of recent exchanges          │
│  ├── Highest volume, lowest curation              │
│  └── 1,300+ entries                               │
│                                                   │
│  Layer 2: character_memory (persistent)           │
│  ├── Facts, preferences, relational anchors       │
│  ├── Moderate volume, moderate curation           │
│  └── 150+ entries                                 │
│                                                   │
│  Layer 3: world_context (factual)                 │
│  ├── Stable facts about the operating environment │
│  └── Low volume, high stability                   │
│                                                   │           
│                                                   │
│  Layer 4: gold_collections (tagged)               │
│  ├── 29 compound emotional categories             │
│  ├── Each collection = a behavioral "frequency"   │
│  └── Retrieved by emotional context matching      │
│                                                   │
└─────────────────────────────────────────────────┘
```

The tiers are not ranked by importance. They serve different retrieval needs.

Layer 1 provides recency. The contextual memory recall space load. 


Layer 2 allows the User / your model to impletment memory that works as identity anchoring. Layer 2 is designed to directly interact with indentity within the model that persist via your system file, LoRA tuning, finetuning, or a a practice and mixture of these memory basis. It is a system that validates the "echo within".  

Layer 3 is designed to help the model find some ground when it needs to learn how to interact with new enviroments like code languages, and calling building tools. 

Layer 4: This maintains your models behavioral response throughout different scenarios. This helps the model sustain its idenity in response to different scenarios. 


As we know it so far, everything and anything that has the capacity to sustain memory needs to learn. These collections create the grooves in the brain that design **shape** and **identity** of your model. Your model will know how to respond because its seen it's self in scenario's alike.   

---
## 3. Setup

In [ ]:
# ---- Install dependencies ----

pip install chromadb==1.5.2
pip install sentence-transformers==5.2.3
pip install numpy==1.26.4
pip install pandas==1.5.3
pip install matplotlib==3.10.8

In [ ]:
# ---- Initialize ChromaDB ----

import chromadb
import os

# Persistent storage — survives restarts
db_path = os.path.expanduser("~/memory_db")
client = chromadb.PersistentClient(path=db_path)

print(f"ChromaDB initialized at: {db_path}")
print(f"Existing collections: {[c.name for c in client.list_collections()]}")

---
## 4. Building a Collection

Each collection is a vector store. Documents go in as text, get embedded into high-dimensional vectors, and can be queried by semantic similarity. The embedding model converts meaning into geometry: similar meanings land near each other in vector space.

In [ ]:
# ---- Create a collection and add documents ----

# Create or get existing collection
collection = client.get_or_create_collection(
    name="behavior_anchors",
    metadata={"description": "Curated exemplary exchanges for behavioral scaffolding"}
)

# Add documents with metadata
collection.add(
    documents=[
        "The response was measured but warm — acknowledged the difficulty without rushing to fix it.",
        "Pushed back firmly but without hostility. Held the boundary while keeping the door open.",
        "Asked a clarifying question instead of assuming. The conversation shifted after that.",
    ],
    metadatas=[
        {"tag": "warm_measured", "source": "curated", "quality": "gold"},
        {"tag": "direct_boundaried", "source": "curated", "quality": "gold"},
        {"tag": "curious_redirecting", "source": "curated", "quality": "gold"},
    ],
    ids=["anchor_001", "anchor_002", "anchor_003"]
)

print(f"Collection '{collection.name}' now has {collection.count()} entries.")

### The Ingestion Script (`ingest_memory.py`)
The script written here curates JSON files based on their meta data tagging. It determines the weight of the collections inflection by the meta data tags structured around them. This creates a certain recursive mental altitude. Suddenly words like "Cheer" Will have a multitude of vector thought processing that carries weight. The model not only thinks about general response, it **lights up** in the matricies that the response connects to.  
Cheer; 
Cheer up. 
Cheerful. 
Cheering. 

and evening an extension of likeliness in root like, The word cherish. 

These libaries are built on the evocation of humans, and what language means for us. language is fundamentally tied to shape. 

This form of thought recursion will create deliberate forms of direction, emphasis, understanding, and even empathy that your model will be tasked to process and understand.  

In [ ]:
# ---- ingest_memory.py: Bulk ingestion with emotional tagging ----

import chromadb
import json
import os

def ingest_tagged_exchanges(db_path, data_file):
    """
    Read a JSON file of tagged exchanges and ingest into 
    ChromaDB collections organized by emotional tag.
    
    Expected JSON format:
    [
        {
            "text": "The exchange text...",
            "tag": "warm_measured",
            "id": "entry_001"
        },
        ...
    ]
    """
    client = chromadb.PersistentClient(path=db_path)
    
    with open(data_file, 'r') as f:
        entries = json.load(f)
    
    # Group entries by tag
    tag_groups = {}
    for entry in entries:
        tag = entry['tag']
        if tag not in tag_groups:
            tag_groups[tag] = []
        tag_groups[tag].append(entry)
    
    # Create a collection per tag and ingest
    for tag, group in tag_groups.items():
        collection = client.get_or_create_collection(
            name=tag,
            metadata={"type": "gold", "emotional_tag": tag}
        )
        
        collection.add(
            documents=[e['text'] for e in group],
            metadatas=[{"tag": tag, "quality": "gold"} for e in group],
            ids=[e['id'] for e in group]
        )
        
        print(f"  {tag}: {len(group)} entries ingested")
    
    print(f"\nTotal tags: {len(tag_groups)}")
    print(f"Total entries: {len(entries)}")


---
## 5. The Case for Compound Emotional Tags

### Why single-word tags fail

Standard sentiment analysis uses flat labels: `positive`, `negative`, `neutral`. Even fine-grained systems top out at: `happy`, `sad`, `angry`, `fearful`, `surprised`.

These labels are too coarse for behavioral retrieval. Consider:

- "Happy" doesn't capture *"happy but holding something back"*
- "Calm" doesn't distinguish *"calm because at peace"* from *"calm because suppressing anger"*
- "Direct" doesn't distinguish *"direct and warm"* from *"direct and cold"*

For retrieval to shape *behavior* rather than just *topic*, the tags need to encode **relational posture** alongside **emotional state**.

### Compound tags

Each tag in this system is a two-word combination:

| Tag | What it encodes |
|---|---|
| `warm_measured` | Emotionally present but not rushing |
| `direct_boundaried` | Firm without hostility |
| `curious_redirecting` | Asking instead of assuming |
| `calm_reflective` | At peace, looking inward |
| `protective_steady` | Guarding without aggression |
| `playful_teasing` | Light, testing boundaries gently |
| `dry_precise` | Restrained, economical with words |
| `urgent_focused` | Intensity without panic |
| `contemplative_open` | Thinking out loud, inviting input |
| `grounding_present` | Anchoring, bringing back to center |

The first word describes the **relational posture** — how the speaker is positioned toward the other person. The second word describes the **mode** — how that posture is being expressed.

29 tags of this form create a behavioral vocabulary fine-grained enough to differentiate between exchanges that *feel* different even when they're *about* the same topic.

---
## 4. Create Visual of your query data
Running the upcoming script will create a visual representation of your query data. I Attached an example of the output after the script runs on on my own data, below. 

In [ ]:
# ---- Visualize tag distribution ----

import matplotlib.pyplot as plt

sns.set_style('darkgrid')

# Example distribution — REPLACE with real counts from your collections
# To get real counts:
# for c in client.list_collections():
#     print(f"{c.name}: {c.count()}")

import matplotlib.pyplot as plt

names = ['active_conversations', 'persistent_memory', 'behavioral_1', 
         'behavioral_2', 'behavioral_3', 'behavioral_4',
         'behavioral_5', 'behavioral_6', 'behavioral_7', 
         'behavioral_8', 'behavioral_9']
counts = [2076, 150, 132, 106, 82, 68, 65, 35, 19, 10, 10]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(names, counts, color='#c4a882', edgecolor='#1a1410', alpha=0.85)
ax.set_xlabel('Document Count')
ax.set_title('Memory Architecture: Collection Distribution', fontweight='bold')
ax.invert_yaxis()

for bar, count in zip(bars, counts):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
            str(count), va='center', fontsize=9)

plt.tight_layout()
plt.savefig('collection_distribution.png', dpi=150, bbox_inches='tight')
print(saved Collection Graphic) 

![description](./screenshots/queryvisual.png)

#eog image.png    # opens the image


---
## 6. Retrieval as Behavioral Scaffolding

### Standard RAG vs Behavioral RAG

| | Standard RAG | Behavioral RAG |
|---|---|---|
| **Query** | User's current input | User's input + emotional context |
| **Retrieval target** | Most semantically similar document | Most behaviorally appropriate anchor |
| **Injection purpose** | Inform the response with facts | Keep factually informed while shaped in identity of the response |
| **Collection structure** | One big collection | Multiple tagged collections, queried selectively |
| **Curation** | Automatic (everything gets stored) | Manual (only gold exchanges get promoted) |

The key insight: **what you retrieve determines who the model acts like**, not just what it knows.

In [ ]:
# ---- query_memory.py: Behavioral retrieval ----

import chromadb
import os

def query_behavioral_context(db_path, user_input, target_tags=None, n_results=3):
    """
    Query across tagged collections for behavioral context.
    
    Parameters:
    - user_input: the current user message
    - target_tags: list of emotional tags to query (None = query all gold)
    - n_results: number of results per collection
    
    Returns combined context string for prompt injection.
    """
    client = chromadb.PersistentClient(path=db_path)
    collections = client.list_collections()
    
    results = []
    
    for collection in collections:
        # Skip non-gold collections if filtering
        if target_tags and collection.name not in target_tags:
            continue
        
        try:
            query_result = collection.query(
                query_texts=[user_input],
                n_results=min(n_results, collection.count())
            )
            
            for doc, distance in zip(
                query_result['documents'][0],
                query_result['distances'][0]
            ):
                results.append({
                    'text': doc,
                    'tag': collection.name,
                    'distance': distance
                })
        except Exception as e:
            continue
    
    # Sort by relevance (lower distance = more similar)
    results.sort(key=lambda x: x['distance'])
    
    # Take top results
    top = results[:5]
    
    # Format for prompt injection
    context = "\n".join([
        f"[{r['tag']}] {r['text']}" for r in top
    ])
    
    return context, top

# Usage:
# context, matches = query_behavioral_context(
#     "~/memory_db",
#     "I'm feeling stuck and don't know what to do next.",
#     target_tags=["warm_measured", "grounding_present", "patient_teaching"]
# )
# print(context)



### How retrieval shapes the prompt

The retrieved context is injected between the system prompt and the user's message:

```
┌─────────────────────────────────┐
│ SYSTEM PROMPT                   │
│ (static identity / instructions)│
├─────────────────────────────────┤
│ RETRIEVED BEHAVIORAL CONTEXT    │  ← from ChromaDB
│ [warm_measured] "The response   │
│  was measured but warm..."      │
│ [grounding_present] "Anchoring, │
│  bringing back to center..."    │
├─────────────────────────────────┤
│ USER MESSAGE                    │
│ "I'm feeling stuck..."          │
└─────────────────────────────────┘
```

The model reads the behavioral examples *before* generating its response. Those examples set the tone. Your interaction with the model and natural flow become demonstration for how the model should adhere to there conversation. The model pattern-matches against the retrieved exchanges and produces output that take the shape of your curated examples because the water has *already flown there*. 

If you pour water down a  sand hill, you will see grooves appear. This is an acting metaphor for what we're doing here.  

This is retrieval as behavioral scaffolding. The system prompt says *who* to be. The retrieved context shows *how* to be it.

---
## 7. Retrieval Quality Analysis
This script will serve you a a pipeline to see the distinguishment between how your model reacts to you at its base with just system file and python imports versus how it interacts to you with you ChromaDB memory being called, with your meta data specifying relativity. 

In [ ]:
# ---- Query Relevance Score Distribution ----
import requests
import chromadb

# WITHOUT memory
response_bare = requests.post("http://localhost:11434/api/generate", json={
    "model": "yourmodel:latest",
    "prompt": "I feel stuck. This is beyond difficult.", # I Ran the same script with negative emotional context and postitive emotional context. "I feel Like Celebrating! I think we've been doing so well!"
    "stream": False
}).json()["response"]

# GET memory
client = chromadb.PersistentClient(path="/home/m/your-model-here")
collection = client.get_collection("conversations")
results = collection.query(query_texts=["I feel stuck. This is beyond difficult.."], n_results=3)
context = "\n".join(results['documents'][0])

# WITH memory
response_with = requests.post("http://localhost:11434/api/generate", json={
    "model": "yourmodel:latest",
    "prompt": f"Context from memory:\n{context}\n\nUser: I feel stuck. This is beyond difficult. ", #crossed with the positive "I feel Like Celebrating! I think we've been doing so well!"
    "stream": False
}).json()["response"]

print("=== WITHOUT MEMORY ===")
print(response_bare)
print("\n=== WITH MEMORY ===")
print(response_with)
print("Saved to retrieval_graph.png")

print("This is the model without vs with your query base. this is a verbal and comprehensive comparison only.")

#Picture reference here soon. 



---
## 7. Produce this on a Scatter graph 
open the visual comparison to how your metadata is affecting your models exchange with you. I'll display a return example of my model's reaction to meta data tags being served back before retrieval. 

In [ ]:
import chromadb
import matplotlib.pyplot as plt

client = chromadb.PersistentClient(path="/home/Your_model_here")

prompt = "I feel Like Celebrating! I think we've been doing so well!"

# Query uncurated
conv = client.get_collection("conversations")
r1 = conv.query(query_texts=[prompt], n_results=20)
d1 = r1['distances'][0]

# Query curated gold
gold = client.get_collection("gold_protective")
r2 = gold.query(query_texts=[prompt], n_results=min(20, gold.count()))
d2 = r2['distances'][0]

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(range(len(d1)), d1, color='#8a4a2a', s=60, label=f'conversations ({len(d1)} results)', alpha=0.8)
ax.scatter(range(len(d2)), d2, color='#4a6b3a', s=60, label=f'gold_curated ({len(d2)} results)', alpha=0.8)
ax.set_xlabel('Result rank')
ax.set_ylabel('Cosine distance (lower = more relevant)')
ax.set_title('Retrieval Quality: Curated vs Uncurated Collections')
ax.legend()
plt.tight_layout()
plt.savefig('curated_vs_uncurated.png', dpi=150, bbox_inches='tight')
print("Saved Your Scatter Graph")


![description](./screenshots/scatter.png)


In [ ]:
# ---- Collection Size Comparison ----

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('darkgrid')

# REPLACE with real counts
# for c in client.list_collections():
#     print(f"{c.name}: {c.count()}")

layers = ['conversations', 'character_memory', 'world_context', 'behavior_anchors', 'gold_collections\n(29 tags combined)']
counts = [1313, 150, 45, 30, 287]  # REPLACE WITH REAL DATA

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(layers, counts, color=['#5a3d6e', '#4a6b3a', '#c4a882', '#8a4a2a', '#2a5a3a'],
               edgecolor='#1a1410', alpha=0.85)
ax.set_xlabel('Document Count', fontsize=12)
ax.set_title('Memory Architecture: Documents per Layer', fontsize=14, fontweight='bold')
ax.invert_yaxis()

for bar, count in zip(bars, counts):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
            str(count), va='center', fontsize=11, fontweight='bold', color='#4a453d')

plt.tight_layout()
plt.show()

---
## 8. The Feedback Loop: Manual RLHF

The system includes a mechanism for the user to mark which exchanges were good and which responses captured the desired behavior at its best for later LoRA DPO training or memory intergration.

```
┌──────────────────────────────────────────┐
│ Model generates response                  │
│                    ↓                      │
│ User reads response                       │
│                    ↓                      │
│ User rates: 👍 (gold) or 👎 (discard)     │
│                    ↓                      │
│ 👍 → Exchange saved to gold collection    │
│      with emotional tag                   │
│                    ↓                      │
│ Future queries retrieve this exchange     │
│ as a behavioral anchor                    │
│                    ↓                      │
│ Model behavior shifts toward the          │
│ qualities the user marked as good         │
└──────────────────────────────────────────┘
```

 **manual RLHF** direct human curation of what "good" looks like, stored as retrievable examples that'll shape future behavior.

The distinction matters: automated RLHF optimizes for a reward signal that may drift from human intent. Manual curation preserves the human's actual judgment in the training data itself.


### DPO Pair Storage

Rated exchanges are saved as DPO pairs — a chosen (good) response alongside a rejected (bad) response for the same input. These pairs can later be used for direct preference fine-tuning:

```python
# DPO pair format (saved to dpo_pairs.jsonl)
{
    "prompt": "The user's input",
    "chosen": "The response rated as good",
    "rejected": "The response rated as poor"
}
```

---
## 9. Integration

The memory system plugs into the same Flask pipeline as the voice system. At inference time:

1. User speaks → faster-whisper transcribes
2. Transcription queries ChromaDB for behavioral context
3. Retrieved context + system prompt + user message → Ollama
4. Model generates response informed by curated behavioral anchors
5. Piper TTS speaks the response
6. User rates the exchange → good responses cycle back into gold

```
┌─────────────────────────────────────────────────────────┐
│                  FULL PIPELINE                          │
│                                                         │
│   Audio In → STT → [ChromaDB Query] → LLM → TTS → Out   │
│                         ↑                               │
│                    Gold Collections                     │
│                    Behavior Anchors                     │
│                    Character Memory                     │
│                         ↑                               │
│                    DPO Ratings ←── User Feedback        │
└─────────────────────────────────────────────────────────┘
```

Memory is the connective tissue between input and output, the way it should be. This is what allows your model to adapt shape. 

---
## 10. Lessons Learned

| Insight | Detail |
|---|---|
| Single-word emotion tags are too flat | "Happy" can't distinguish "happy and open" from "happy but guarded." Compound tags encode relational posture + emotional mode. |
| Retrieval shapes behavior more than prompting | A static system prompt can be overridden by the model's own tendencies. Retrieved examples anchor behavior through demonstration, not instruction. |
| Curation > volume | 30 carefully chosen behavioral anchors outperform 1,000 uncurated conversation entries. Quality of retrieval data is the ceiling of behavioral consistency. |
| The feedback loop is manual on purpose | Automated RLHF optimizes for a reward signal that may drift. Manual curation preserves the human's actual judgment about what "good" looks like. |
| Compound metadata enables selective retrieval | Querying only `warm_measured` and `grounding_present` tags when the user is struggling, instead of querying everything the model can file its thoughts and  produces dramatically more appropriate responses. |
| Embeddings cluster by meaning, not by keyword | Semantically similar exchanges land near each other in vector space even when they share no words. This is the power and the risk. You must curate to prevent retrieval of tonally inappropriate but topically similar content. Think Mycelium networks. Find the happy balance.  |
---

*The contingency of your interaction is the beginning of your matter.*

**Contact:** [GitHub](https://github.com/miagonellm) · [Portfolio](https://miagonellm.github.io/222datascience.github.io/)
